# [M5] CHIẾN THUẬT THI OLYMPIC AI
## Buổi S01: Chiến Thuật Đọc Đề Nhanh, Xác Định Target, Metric & Phòng Ngừa Data Leakage

---

### Metadata Khóa Học
* **Module:** M5 — CHIẾN THUẬT THI OLYMPIC AI
* **Buổi:** S01
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook kiểm toán đề thi và thiết lập khung Validation an toàn
* **Mức độ liên quan VOAI / Olympic AI:** Bảo hiểm số 1 chống lại hiện tượng tụt hạng (Shakeup) trên bảng xếp hạng Olympic.

#### Mục tiêu học tập (Learning Objectives)
- Làm chủ quy trình '15 phút đọc đề & giải mã bài toán thi đấu' (Problem Deconstruction Playbook).
- Xác định bản chất phân phối của biến mục tiêu (Target Analysis) và các rủi ro tiềm ẩn (nhãn hiếm, mất cân bằng cực đoan).
- Căn chỉnh hàm mất mát khi huấn luyện (Loss Function) khớp chính xác 100% với Metric chấm thi chính thức của Ban tổ chức.
- Thiết lập sơ đồ kiểm chứng chéo (Cross-Validation Strategy) chuẩn hóa chống rò rỉ dữ liệu (Data Leakage Safeguards).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành các module M1, M2, M3, M4.
- Hiểu sâu về các chỉ số đánh giá mô hình (Precision, Recall, F1, RMSE) từ M2_S04.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong một kỳ thi Olympic AI hoặc giải đấu Competitive AI, thời gian làm bài thường chỉ từ 3 đến 4 giờ liên tục. Sai lầm phổ biến nhất của 80% thí sinh thất bại là: **Vừa mở đề ra đã vội vàng lao vào gõ code, nạp các thư viện phức tạp mà không đọc kỹ thể lệ và cấu trúc dữ liệu**.

Hậu quả là sau 3 tiếng cày cuốc, thí sinh mới ngã ngửa phát hiện: Đề bài yêu cầu tối ưu Macro-F1 nhưng mình lại huấn luyện mô hình theo Accuracy; hoặc dữ liệu có tính chất nhóm (Group Data) nhưng mình chia ngẫu nhiên dẫn đến rò rỉ dữ liệu (Data Leakage) nghiêm trọng. 15 phút đầu tiên quyết định 80% thắng bại của một bài thi. Làm chủ chiến thuật đọc đề và phòng ngừa rò rỉ dữ liệu là tố chất phân định giữa một người chơi nghiệp dư và một vận động viên săn giải!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Quy trình 15 phút đọc đề chuẩn Olympic (15-Minute Playbook)
* **Phút 01 - 03 (Hiểu Đích đến):** Xác định chính xác **Metric chấm thi chính thức** (Accuracy, Log-Loss, Macro-F1, ROC-AUC, hay RMSE) và định dạng file nộp bài (`submission.csv` gồm những cột nào?).
* **Phút 04 - 07 (Khám phá Biến Mục tiêu - Target):**
  - Là bài toán gì: Hồi quy (Regression) hay Phân loại (Classification)?
  - Phân phối nhãn có cân bằng không? (Nếu lớp 1 chỉ chiếm 2%, lập tức bật chế độ xử lý mất cân bằng).
* **Phút 08 - 11 (Kiểm toán Dữ liệu - Data Audit):**
  - Kích thước tập Train và tập Test ($N_{train}, N_{test}$, số cột).
  - Có cột thời gian không? (Nếu có $\implies$ Bắt buộc dùng Time-Series Split, cấm Random Split).
  - Có cột định danh nhóm (`User_ID`, `Patient_ID`, `Shop_ID`) không? (Nếu có $\implies$ Bắt buộc dùng GroupKFold).
* **Phút 12 - 15 (Thiết lập Validation Scheme):** Dựng khung kiểm thử nội bộ khớp với hệ thống chấm ẩn.

### 2. Ba cạm bẫy Rò rỉ Dữ liệu (Data Leakage Traps) thường gặp
1. **Temporal Leakage (Rò rỉ thời gian):** Dùng dữ liệu tương lai để dự đoán quá khứ.
2. **Group Leakage (Rò rỉ nhóm):** Cùng một bệnh nhân có 5 tấm ảnh chụp, 3 tấm nằm ở Train và 2 tấm nằm ở Validation $\implies$ Mô hình nhận diện bệnh nhân thay vì nhận diện bệnh!
3. **Preprocessing Leakage (Rò rỉ tiền xử lý):** Fit Scaler hoặc Imputer trên toàn bộ dataset trước khi chia fold.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Chiến thuật Lựa chọn Sơ đồ Validation theo đặc thù dữ liệu:
                     ┌── Dữ liệu có yếu tố Thời gian? ────────────► [Time-Series Split / Temporal Split]
                     │
[TẬP DỮ LIỆU ĐỀ THI] ├── Có các mẫu thuộc cùng một Đối tượng? ───► [GroupKFold (Không để trùng ID giữa các Fold)]
                     │
                     └── Dữ liệu Bảng độc lập thông thường? ───────► [StratifiedKFold (Giữ nguyên tỷ lệ nhãn)]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GroupKFold

# 1. Kich ban thi dau: Phat hien benh nhan co nguy co dot quy (Co yeu to Group va Imbalanced)
np.random.seed(42)
n_records = 300

# Gia lap 60 benh nhan, moi benh nhan duoc do suc khoe 5 lan (co cot Patient_ID)
patient_ids = np.repeat(np.arange(1, 61), 5)
np.random.shuffle(patient_ids)

# Dac trung: Huyet ap, Tuoi va Nhan dot quy (Target: 1 la Dot quy, chi chiem 10%)
age = np.random.randint(40, 85, size=n_records)
bp = np.random.normal(135, 20, size=n_records).round(1)
stroke = np.random.choice([0, 1], size=n_records, p=[0.9, 0.1])

df_exam = pd.DataFrame({
    'Record_ID': np.arange(1, n_records + 1),
    'Patient_ID': patient_ids,
    'Age': age,
    'Blood_Pressure': bp,
    'Stroke_Risk': stroke
})

print("===== 15 PHUT DOC DE: KIEM TOAN DU LIEU THI DAU =====")
print(f"Kich thuoc tap du lieu: {df_exam.shape[0]} dong, {df_exam.shape[1]} cot")
print(f"Ty le nhan dot quy (Stroke=1): {df_exam['Stroke_Risk'].mean()*100:.1f}% (Mat can bang!)")
print(f"So luong benh nhan duy nhat: {df_exam['Patient_ID'].nunique()} benh nhan tren 300 ban ghi")
print("-> PHAN TICH CHIEN THUAT: Co su trung lap Patient_ID giua cac ban ghi. BAT BUOC DUNG GroupKFold de tranh Group Leakage!")

### 🔍 Code Walkthrough (Khối 1)
Khối code mô phỏng 15 phút đọc đề và kiểm toán dữ liệu thi đấu:
- Nhanh chóng định lượng kích thước, tỷ lệ nhãn lệch (10%).
- Phát hiện ra yếu tố nhóm then chốt: 300 bản ghi nhưng chỉ thuộc về 60 bệnh nhân (`Patient_ID`).
- Đưa ra quyết định chiến thuật chính xác: Phải dùng `GroupKFold` để ngăn ngừa rò rỉ dữ liệu giữa các fold.


In [ ]:
# 2. Thiet lap Sơ đồ Validation GroupKFold chuan phong thi Olympic
gkf = GroupKFold(n_splits=5)
X = df_exam[['Age', 'Blood_Pressure']]
y = df_exam['Stroke_Risk']
groups = df_exam['Patient_ID']

print("===== KIEM CHUNG SO DO CHIA 5-FOLD GROUP VALIDATION =====")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    train_patients = set(df_exam.iloc[train_idx]['Patient_ID'])
    val_patients = set(df_exam.iloc[val_idx]['Patient_ID'])
    
    # Kiem tra xem co benh nhan nao xuat hien o ca 2 tap khong (Giao nhau phai la rong)
    intersection = train_patients.intersection(val_patients)
    print(f"Fold {fold}: Train size = {len(train_idx)} | Val size = {len(val_idx)} | Trun lap ID: {len(intersection)} benh nhan")

print("\n-> XAC NHAN: Khong co bat ky benh nhan nao bi trung lap giua Train va Val. So do kiem chung an toan 100%!")

### 🔍 Code Walkthrough (Khối 2)
Khối code thiết lập và kiểm chứng sơ đồ `GroupKFold`:
- Chia đều 60 bệnh nhân vào 5 fold độc lập (mỗi fold chứa dữ liệu của 12 bệnh nhân hoàn toàn mới).
- Xác minh bằng thuật toán tập hợp: Số lượng bệnh nhân giao nhau bằng đúng 0 (`len(intersection) == 0`).
- Bảo đảm điểm số validation nội bộ sẽ tương thích trung thực với điểm số Private Leaderboard.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Phân tích Metric thi đấu
Nếu đề thi yêu cầu metric đánh giá là **Quadratic Weighted Kappa (QWK)** (thường dùng trong chấm điểm tự luận hoặc phân độ bệnh từ 0 đến 4), hãy tìm hiểu xem QWK phạt các sai số lệch xa (ví dụ thực tế là độ 0 nhưng đoán nhầm là độ 4) nặng nề như thế nào so với Accuracy thông thường.

### Bài 2 (Easy): Phát hiện Data Leakage trong mã nguồn đối thủ
Đoạn code sau đây có lỗi Data Leakage ở bước nào?
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) # Chuẩn hóa toàn bộ trước
for train_idx, val_idx in kfold.split(X_scaled, y):
    model.fit(X_scaled[train_idx], y[train_idx])
```
Hãy viết lại đoạn code trên đúng quy cách an toàn.

### Bài 3 (Medium): Sơ đồ StratifiedGroupKFold
Trong nhiều bài toán thực tế, dữ liệu vừa có tính chất Nhóm (`Group`) vừa bị Mất cân bằng nhãn (`Imbalanced`). Tìm hiểu lớp `StratifiedGroupKFold` trong Scikit-Learn và giải thích cách nó cân bằng đồng thời cả hai yếu tố này.

### Bài 4 (Medium): Xây dựng hàm kiểm toán dữ liệu 1-Click (1-Click Data Auditor)
Viết một hàm Python `audit_competition_data(train_df, test_df, target_col)` tự động in ra màn hình một bản tóm tắt chiến thuật gồm: số hàng, số cột, tỷ lệ nhãn, số lượng cột có missing values và danh sách các cột ID nghi ngờ.

### Bài 5 (Challenge): Mô phỏng hiện tượng "Rớt hạng Private Leaderboard" (Shakeup Simulator)
Viết một kịch bản giả lập: Huấn luyện một mô hình bị Overfit/Leakage và một mô hình có Validation chuẩn. Chứng minh rằng mô hình bị rò rỉ có điểm Public rất cao nhưng khi chuyển sang tập Private ẩn thì thứ hạng bị tụt dốc thảm hại (Shakeup rớt 50 bậc).



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong lịch sử các kỳ thi Olympic AI, hiện tượng **Shakeup** (xáo trộn bảng tổng sắp khi công bố điểm Private) là nỗi ám ảnh lớn nhất: Nhiều đội tuyển đứng vị trí Top 3 trên bảng điểm mở (Public Leaderboard) nhưng khi ban giám khảo mở tập dữ liệu kiểm tra ẩn (Private Test) thì rơi thẳng xuống vị trí số 40 vì mô hình bị Overfit do chia fold cẩu thả. Ngược lại, những thí sinh có sơ đồ Validation chặt chẽ, không rò rỉ dữ liệu sẽ vững vàng tiến thẳng tới tấm Huy chương Vàng.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh nộp file dự đoán nhưng bị hệ thống chấm thi tự động từ chối với thông báo lỗi `Submission Format Error`:

```python
# CODE CÓ LỖI QUY CÁCH SUBMISSION:
# De bai yeu cau: File submission.csv gom 2 cot ['ID', 'Prediction'] va 1000 dong.
# Thi sinh xuat file bang Pandas:
# df_sub.to_csv("submission.csv") # Quen tham so index=False!
# File tao ra se co them cot index khong ten o dau tien -> He thong cham tu dong bao loi ngay lap tuc!
```

**Nhiệm vụ của bạn:**
1. Chỉ ra tại sao luôn phải có `index=False` khi xuất file nộp bài bằng `to_csv`.
2. Viết đoạn code kiểm tra kiểm thử (Sanity Check) so sánh shape và tên cột của file submission với file mẫu `sample_submission.csv`.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Trong 15 phút đầu tiên của một bài thi Olympic AI, hành động nào sau đây là QUAN TRỌNG NHẤT?
   - A. Mở nhạc thư giãn.
   - B. Đọc kỹ đề bài, xác định chính xác Metric chấm thi, phân tích phân phối nhãn và kiểm toán các cạm bẫy rò rỉ dữ liệu để dựng sơ đồ Validation chuẩn.
   - C. Cài đặt các thư viện mới nhất từ GitHub.
   - D. Viết ngay một mạng Deep Learning 50 tầng.

2. Hiện tượng "Group Data Leakage" xảy ra khi nào?
   - A. Khi máy tính bị mất điện đột ngột.
   - B. Khi dữ liệu của cùng một đối tượng (ví dụ: cùng một bệnh nhân hoặc cùng một khách hàng) xuất hiện đồng thời ở cả tập Huấn luyện (Train) và tập Kiểm chứng (Validation).
   - C. Khi file CSV có quá nhiều cột.
   - D. Khi dùng thư viện Scikit-Learn phiên bản cũ.

3. Sơ đồ chia tập nào là bắt buộc khi dữ liệu có yếu tố chuỗi thời gian (Time-Series Data, ví dụ: giá cổ phiếu từng ngày)?
   - A. K-Fold xáo trộn ngẫu nhiên hoàn toàn (Random K-Fold).
   - B. Time-Series Split (Cắt theo trình tự thời gian: Quá khứ dùng để học, Tương lai dùng để kiểm chứng).
   - C. Chọn ngẫu nhiên 50% dữ liệu.
   - D. Không cần chia tập kiểm chứng.

4. Lệnh xuất file nộp bài chuẩn quy cách trong Pandas để không sinh thêm cột chỉ số thừa là gì?
   - A. `df.to_csv('submission.csv')`
   - B. `df.to_csv('submission.csv', index=False)`
   - C. `df.save('submission.csv')`
   - D. `df.export_csv()`

5. Hiện tượng "Shakeup" trên bảng xếp hạng thi đấu AI diễn ra khi nào?
   - A. Khi hệ thống chấm thi bị sập máy chủ.
   - B. Khi kết quả thi đấu chính thức được công bố trên tập kiểm tra ẩn (Private Leaderboard), thứ hạng của các thí sinh bị đảo lộn dữ dội so với điểm số tạm thời trên Public Leaderboard do nhiều mô hình bị Overfitting.
   - C. Khi thí sinh nộp bài muộn hơn quy định.
   - D. Khi ban tổ chức đổi đề thi giữa giờ.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Quy trình 15 phút đọc đề và giải mã bài toán thi đấu đỉnh cao.
  - Ba cạm bẫy rò rỉ dữ liệu (Temporal, Group, Preprocessing Leakage).
  - Cách thiết lập sơ đồ kiểm chứng chéo (StratifiedKFold, GroupKFold) an toàn tuyệt đối.
* **What I can now do:**
  - Kiểm toán nhanh bất kỳ tập dữ liệu thi đấu nào trong chưa đầy 10 phút.
  - Xây dựng hệ thống Validation đáng tin cậy, bảo vệ thành quả khỏi hiện tượng Shakeup.
* **What I still need to practice:**
  - Nâng cấp điểm số bằng **Feature Engineering Thực Chiến & K-Fold Ensembling** ở Buổi 2.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M5_S02**, chúng ta sẽ bước vào vũ khí tạo ra sự khác biệt lớn nhất về điểm số: **Feature Engineering Thực Chiến, K-Fold Validation & Tối Ưu Hóa Điểm Số** — Khám phá kỹ thuật tạo biến tương tác, Target Encoding chống rò rỉ, và phối hợp mô hình (Blending/Averaging)!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M5**:
- Chiến thuật 15 phút đọc đề & Target analysis ✓
- Căn chỉnh Loss function khớp Metric thi đấu ✓
- Sơ đồ Cross-Validation (GroupKFold, StratifiedKFold) chống Data Leakage ✓

* **Session Deliverable:** Mã nguồn kiểm toán dữ liệu và thiết lập sơ đồ Validation phòng thi an toàn
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
